In [1]:
"""
한국 시장총액 상위 30개 기업의 일간 주가 수익률 데이터프레임 생성

실행 환경: 로컬 PC, Google Colab, Jupyter Notebook 등
필요 라이브러리: pip install finance-datareader pandas
"""

import FinanceDataReader as fdr
import pandas as pd
from datetime import datetime, timedelta

def get_top30_stock_returns():
    """
    한국 시장총액 상위 30개 기업의 최근 1년간 일간 수익률 데이터프레임 생성

    Returns:
        pd.DataFrame: date를 index로, 종목별 일간 수익률을 값으로 하는 데이터프레임
    """

    # 1. 한국 거래소 상장 종목 리스트 가져오기
    print("한국 거래소 상장 종목 데이터 로딩 중...")
    krx_stocks = fdr.StockListing('KRX')

    # 2. 시가총액 기준 상위 30개 종목 선택
    print("시가총액 상위 30개 종목 선택 중...")
    top30_stocks = krx_stocks.nlargest(30, 'Marcap')[['Code', 'Name', 'Marcap']]

    print("\n시가총액 상위 30개 종목:")
    print(top30_stocks.to_string())

    # 3. 최근 1년 날짜 설정
    end_date = datetime.now()
    start_date = end_date - timedelta(days=365)

    print(f"\n데이터 수집 기간: {start_date.date()} ~ {end_date.date()}")
    print("-" * 60)

    # 4. 각 종목의 주가 데이터 수집 및 일간 수익률 계산
    returns_df = pd.DataFrame()
    success_count = 0
    failed_stocks = []

    for idx, row in top30_stocks.iterrows():
        code = row['Code']
        name = row['Name']

        try:
            print(f"Processing: {name} ({code})...", end=' ')

            # 주가 데이터 가져오기
            stock_data = fdr.DataReader(code, start_date, end_date)

            if stock_data.empty:
                print("⚠️ 데이터 없음")
                failed_stocks.append((code, name))
                continue

            # 일간 수익률 계산 (Close 기준)
            # pct_change(): (오늘 가격 / 어제 가격) - 1
            daily_returns = stock_data['Close'].pct_change()

            # 데이터프레임에 추가 (컬럼명은 종목명)
            returns_df[name] = daily_returns

            success_count += 1
            print(f"✓ ({len(stock_data)}일)")

        except Exception as e:
            print(f"✗ Error: {str(e)}")
            failed_stocks.append((code, name))
            continue

    # 5. 첫 번째 행(NaN) 제거 - pct_change()의 첫 값은 항상 NaN
    returns_df = returns_df.dropna(how='all')

    # 6. 결과 요약 출력
    print("\n" + "=" * 60)
    print("데이터 수집 완료!")
    print("=" * 60)
    print(f"✓ 성공: {success_count}개 종목")
    print(f"✗ 실패: {len(failed_stocks)}개 종목")

    if failed_stocks:
        print("\n실패 종목:")
        for code, name in failed_stocks:
            print(f"  - {name} ({code})")

    print(f"\n📊 데이터프레임 정보:")
    print(f"  - Shape: {returns_df.shape}")
    print(f"  - 기간: {returns_df.index[0].date()} ~ {returns_df.index[-1].date()}")
    print(f"  - 총 거래일 수: {len(returns_df)}일")
    print(f"  - 총 종목 수: {len(returns_df.columns)}개")

    return returns_df


def analyze_returns(returns_df):
    """수익률 데이터 기초 분석"""

    print("\n" + "=" * 60)
    print("📈 최근 5일 일간 수익률 (%):")
    print("=" * 60)
    print((returns_df.tail() * 100).round(2))

    print("\n" + "=" * 60)
    print("📊 기초 통계량:")
    print("=" * 60)

    # 평균 수익률
    mean_returns = returns_df.mean() * 100
    print("\n평균 일간 수익률 상위 10개 (%):")
    print(mean_returns.sort_values(ascending=False).head(10).round(3))

    # 변동성 (표준편차)
    std_returns = returns_df.std() * 100
    print("\n변동성(표준편차) 상위 10개 (%):")
    print(std_returns.sort_values(ascending=False).head(10).round(3))

    # 누적 수익률
    cumulative_returns = (1 + returns_df).prod() - 1
    print("\n누적 수익률 상위 10개 (%):")
    print((cumulative_returns * 100).sort_values(ascending=False).head(10).round(2))

    # 상관관계 행렬
    print("\n📊 종목 간 평균 상관계수:")
    corr_matrix = returns_df.corr()
    # 대각선 제외한 평균
    mask = ~pd.np.eye(corr_matrix.shape[0], dtype=bool)
    avg_corr = corr_matrix.where(mask).mean().mean()
    print(f"  {avg_corr:.3f}")


# 메인 실행 부분
# if __name__ == "__main__":
#     # 1. 데이터 수집
#     returns_df = get_top30_stock_returns()
#
#     # 2. 기초 분석
#     analyze_returns(returns_df)
#
#     # 3. CSV 파일로 저장
#     output_file = 'korea_top30_daily_returns.csv'
#     returns_df.to_csv(output_file)
#     print(f"\n💾 데이터 저장 완료: {output_file}")
#
#     # 4. 사용 예시
#     print("\n" + "=" * 60)
#     print("📝 데이터프레임 사용 예시:")
#     print("=" * 60)
#     print("""
# # 특정 종목의 수익률 데이터
# samsung_returns = returns_df['삼성전자']
#
# # 특정 날짜의 모든 종목 수익률
# date_returns = returns_df.loc['2024-11-15']
#
# # 특정 종목의 평균 수익률
# avg_return = returns_df['삼성전자'].mean()
#
# # 전체 포트폴리오 일별 평균 수익률
# portfolio_returns = returns_df.mean(axis=1)
#
# # 결측치 확인
# print(returns_df.isnull().sum())
#
# # 기초 통계량
# print(returns_df.describe())
#     """)
#
#     print("\n✅ 모든 작업 완료!")
#
#     # 데이터프레임 반환
#     # returns_df 변수를 통해 후속 분석 가능
#
#
# # ==========================================
# # 간단한 버전 (함수 없이)
# # ==========================================
# """
# import FinanceDataReader as fdr
# import pandas as pd
# from datetime import datetime, timedelta
#
# # KRX 상장 종목 리스트
# krx = fdr.StockListing('KRX')
#
# # 시총 상위 30개
# top30 = krx.nlargest(30, 'Marcap')
#
# # 날짜 설정
# end = datetime.now()
# start = end - timedelta(days=365)
#
# # 수익률 데이터프레임
# returns_df = pd.DataFrame()
#
# # 데이터 수집
# for _, row in top30.iterrows():
#     try:
#         data = fdr.DataReader(row['Code'], start, end)
#         returns_df[row['Name']] = data['Close'].pct_change()
#     except:
#         pass
#
# # NaN 제거
# returns_df = returns_df.dropna(how='all')
#
# # 저장
# returns_df.to_csv('returns.csv')
# print(returns_df)
# """

In [2]:
returns_df = get_top30_stock_returns()
returns_df

한국 거래소 상장 종목 데이터 로딩 중...
시가총액 상위 30개 종목 선택 중...

시가총액 상위 30개 종목:
      Code       Name           Marcap
0   005930       삼성전자  578940588771600
1   000660     SK하이닉스  414961348050000
2   373220   LG에너지솔루션  103662000000000
3   207940   삼성바이오로직스   86903454000000
4   005935      삼성전자우   60463722602400
5   005380        현대차   54158429107000
6   329180    HD현대중공업   53530188948000
7   034020    두산에너빌리티   48298310408400
8   012450  한화에어로스페이스   47489892321000
9   105560       KB금융   46729107617500
10  000270         기아   45088871415000
11  068270       셀트리온   42450626102200
12  042660       한화오션   39159631753200
13  035420      NAVER   39056306862000
14  402340      SK스퀘어   37575333243000
15  055550       신한지주   37091812957600
16  028260       삼성물산   36459968688000
17  015760       한국전력   31777221811500
18  009540   HD한국조선해양   30149347416000
19  196170       알테오젠   29909735492000
20  032830       삼성생명   29740000000000
21  267260   HD현대일렉트릭   29270273620000
22  051910       LG화학   27389829084000

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,삼성전자우,현대차,HD현대중공업,두산에너빌리티,한화에어로스페이스,KB금융,...,삼성생명,HD현대일렉트릭,LG화학,현대모비스,카카오,하나금융지주,POSCO홀딩스,삼성SDI,삼성중공업,삼성화재
Date,,,,,,,,,,,,,,,,,,,,,
2024-11-20,-0.017762,0.000000,0.017789,0.020675,-0.014523,0.013921,0.051225,-0.031250,-0.002623,0.043668,...,0.003817,-0.006766,0.010399,0.017964,-0.020690,0.020033,0.013746,0.001932,0.012766,0.026063
2024-11-21,0.019892,-0.010551,0.000000,-0.006397,0.027368,-0.013730,-0.031780,-0.036866,-0.061843,0.017782,...,-0.005703,-0.021798,0.010292,-0.003922,-0.009859,0.008183,0.030508,0.019267,-0.007563,-0.006684
2024-11-22,-0.007092,0.046801,0.011236,0.002146,-0.014344,0.006961,0.037199,0.057416,0.064516,0.011305,...,0.018164,0.011142,0.011885,0.000000,0.025605,0.014610,-0.001645,0.017011,0.018628,0.025572
2024-11-25,0.033929,0.001698,0.035802,0.025696,0.019751,0.009217,0.021097,0.002262,-0.054017,-0.004065,...,-0.004695,0.064738,0.028523,-0.029528,0.002774,0.006400,-0.006590,0.031599,-0.009975,-0.015748
2024-11-26,0.006908,0.000565,-0.032181,-0.030271,-0.001019,0.020548,-0.055785,-0.049661,-0.110028,-0.020408,...,-0.013208,-0.055627,0.001631,0.010142,0.023513,-0.012719,0.003317,0.003603,-0.007557,-0.013333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-11-12,-0.003865,-0.003231,0.005285,0.000000,-0.003802,0.024164,0.003745,0.002558,0.012658,0.030604,...,0.047679,-0.013953,-0.015019,0.005025,-0.004680,0.037675,0.051155,-0.001553,-0.005682,0.041916
2025-11-13,-0.002910,-0.008104,0.018927,0.000000,0.008906,0.010889,0.059701,0.059949,-0.004167,-0.008909,...,0.008383,0.021226,0.066074,0.000000,-0.001567,-0.002075,0.009419,0.040435,0.009524,-0.042146
2025-11-14,-0.054475,-0.084967,-0.044376,0.000000,-0.058008,-0.021544,0.031690,-0.056558,-0.001046,-0.029963,...,-0.087886,-0.048499,-0.028605,-0.006667,-0.036107,-0.018711,-0.023328,-0.058296,-0.007547,-0.011000


In [3]:
"""
최소분산 포트폴리오 계산 - 간단한 버전
CSV 파일에서 수익률 데이터를 읽어 최적 투자비중 계산
"""

import pandas as pd
import numpy as np
from scipy.optimize import minimize

# ==========================================
# 1. 데이터 로드
# ==========================================
# print("=" * 70)
# print("📂 데이터 로딩...")
# print("=" * 70)

# # CSV 파일 경로 (실제 파일명으로 변경)
# csv_file = 'korea_top30_daily_returns.csv'  # 여기에 실제 파일명 입력
#
# # 데이터 로드
# returns_df = pd.read_csv(csv_file, index_col=0, parse_dates=True)
#
# print(f"✅ 데이터 로드 완료")
# print(f"  - 종목 수: {len(returns_df.columns)}개")
# print(f"  - 기간: {returns_df.index[0]} ~ {returns_df.index[-1]}")
# print(f"  - 총 데이터 수: {len(returns_df)}일")

returns_df = get_top30_stock_returns()

# 결측치 처리
if returns_df.isnull().any().any():
    print(f"\n⚠️ 결측치 발견 - 0으로 대체")
    returns_df = returns_df.fillna(0)

print(f"\n최근 5일 수익률 샘플:")
print(returns_df.tail())

# ==========================================
# 2. 최소분산 포트폴리오 최적화
# ==========================================
print("\n" + "=" * 70)
print("⏳ 최소분산 포트폴리오 최적화 진행 중...")
print("=" * 70)

# 종목 수
n_assets = len(returns_df.columns)

# 공분산 행렬 계산 (연율화)
cov_matrix = returns_df.cov() * 252

print(f"공분산 행렬 크기: {cov_matrix.shape}")

# 목적함수: 포트폴리오 분산
def portfolio_variance(weights):
    return np.dot(weights.T, np.dot(cov_matrix, weights))

# 제약조건: 비중의 합 = 1
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

# 경계조건: 0 <= 비중 <= 1 (공매도 불가)
bounds = tuple((0, 1) for _ in range(n_assets))

# 초기값: 균등 배분
initial_weights = np.array([1/n_assets] * n_assets)

# 최적화 실행
result = minimize(
    portfolio_variance,
    initial_weights,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-9}
)

# 최적 비중
optimal_weights = result.x

if result.success:
    print("✅ 최적화 성공!")
else:
    print(f"⚠️ 최적화 경고: {result.message}")

# ==========================================
# 3. 포트폴리오 성과 계산
# ==========================================
print("\n" + "=" * 70)
print("📈 포트폴리오 성과 지표 (연율화)")
print("=" * 70)

# 평균 수익률 (연율화)
mean_returns = returns_df.mean() * 252

# 포트폴리오 기대수익률
portfolio_return = np.dot(optimal_weights, mean_returns)

# 포트폴리오 분산
portfolio_variance_value = np.dot(optimal_weights.T, np.dot(cov_matrix, optimal_weights))

# 포트폴리오 표준편차 (변동성)
portfolio_std = np.sqrt(portfolio_variance_value)

print(f"기대 수익률:      {portfolio_return*100:6.2f}%")
print(f"변동성(표준편차): {portfolio_std*100:6.2f}%")
print(f"분산:            {portfolio_variance_value:6.4f}")

# 샤프 비율 (무위험수익률 = 0 가정)
sharpe_ratio = portfolio_return / portfolio_std
print(f"샤프 비율:        {sharpe_ratio:6.3f}")

# ==========================================
# 4. 최적 투자비중 출력
# ==========================================
print("\n" + "=" * 70)
print("💼 최소분산 포트폴리오 - 최적 투자비중")
print("=" * 70)

# 데이터프레임 생성
weights_df = pd.DataFrame({
    '종목명': returns_df.columns,
    '투자비중': optimal_weights,
    '투자비중(%)': optimal_weights * 100
}).sort_values('투자비중', ascending=False)

# 전체 비중 출력
print("\n[전체 종목 투자비중]")
print("-" * 70)
for idx, row in weights_df.iterrows():
    if row['투자비중(%)'] >= 0.01:  # 0.01% 이상만 막대 표시
        bar_length = int(row['투자비중(%)'] / 2)
        bar = '█' * bar_length
        print(f"{row['종목명']:20s} │ {row['투자비중(%)']:7.3f}% │ {bar}")
    else:
        print(f"{row['종목명']:20s} │ {row['투자비중(%)']:7.3f}%")

print("-" * 70)
print(f"{'합계':20s} │ {weights_df['투자비중(%)'].sum():7.3f}%")

# 주요 종목 (비중 1% 이상)
significant_weights = weights_df[weights_df['투자비중(%)'] >= 1.0]
print(f"\n📊 비중 1% 이상 종목: {len(significant_weights)}개")
print(f"   총 비중: {significant_weights['투자비중(%)'].sum():.2f}%")

# 상위 10개 종목
print(f"\n🏆 상위 10개 종목:")
print(weights_df.head(10)[['종목명', '투자비중(%)']].to_string(index=False))

# ==========================================
# 5. 개별 종목과 비교
# ==========================================
print("\n" + "=" * 70)
print("📊 분산 효과 분석")
print("=" * 70)

# 개별 종목 통계
individual_returns = returns_df.mean() * 252
individual_stds = returns_df.std() * np.sqrt(252)

print(f"\n개별 종목 평균:")
print(f"  - 평균 수익률: {individual_returns.mean()*100:6.2f}%")
print(f"  - 평균 변동성: {individual_stds.mean()*100:6.2f}%")
print(f"  - 최소 변동성: {individual_stds.min()*100:6.2f}% ({individual_stds.idxmin()})")
print(f"  - 최대 변동성: {individual_stds.max()*100:6.2f}% ({individual_stds.idxmax()})")

print(f"\n최소분산 포트폴리오:")
print(f"  - 수익률: {portfolio_return*100:6.2f}%")
print(f"  - 변동성: {portfolio_std*100:6.2f}%")

# 분산 효과
volatility_reduction = (1 - portfolio_std / individual_stds.mean()) * 100
print(f"\n✨ 분산 효과: 평균 대비 변동성 {volatility_reduction:.1f}% 감소")

# 최소 변동성 종목과 비교
min_vol_stock_std = individual_stds.min()
if portfolio_std < min_vol_stock_std:
    print(f"   개별 종목 최소 변동성보다도 {(1-portfolio_std/min_vol_stock_std)*100:.1f}% 낮음!")

# ==========================================
# 6. 결과 저장
# ==========================================
# output_file = 'minimum_variance_portfolio_weights.csv'
# weights_df.to_csv(output_file, index=False, encoding='utf-8-sig')
# print(f"\n💾 결과 저장: {output_file}")

# ==========================================
# 7. 투자 가이드
# ==========================================
print("\n" + "=" * 70)
print("📝 투자 실행 가이드")
print("=" * 70)
print("\n1. 투자금액 계산 예시 (총 1억원 투자 시):")
print("-" * 70)

investment_amount = 100_000_000  # 1억원
top5_weights = weights_df.head(5)

for idx, row in top5_weights.iterrows():
    amount = investment_amount * row['투자비중']
    print(f"{row['종목명']:20s} │ {row['투자비중(%)']:6.2f}% │ {amount:>15,.0f}원")

print("\n2. 리밸런싱 주기: 분기별 또는 비중 5%p 이상 변동 시")
print("3. 모니터링: 월별 성과 및 위험지표 점검")
print("4. 백테스팅: 과거 데이터로 전략 검증 권장")

print("\n✅ 분석 완료!")

한국 거래소 상장 종목 데이터 로딩 중...
시가총액 상위 30개 종목 선택 중...

시가총액 상위 30개 종목:
      Code       Name           Marcap
0   005930       삼성전자  578940588771600
1   000660     SK하이닉스  414961348050000
2   373220   LG에너지솔루션  103662000000000
3   207940   삼성바이오로직스   86903454000000
4   005935      삼성전자우   60463722602400
5   005380        현대차   54158429107000
6   329180    HD현대중공업   53530188948000
7   034020    두산에너빌리티   48298310408400
8   012450  한화에어로스페이스   47489892321000
9   105560       KB금융   46729107617500
10  000270         기아   45088871415000
11  068270       셀트리온   42450626102200
12  042660       한화오션   39159631753200
13  035420      NAVER   39056306862000
14  402340      SK스퀘어   37575333243000
15  055550       신한지주   37091812957600
16  028260       삼성물산   36459968688000
17  015760       한국전력   31777221811500
18  009540   HD한국조선해양   30149347416000
19  196170       알테오젠   29909735492000
20  032830       삼성생명   29740000000000
21  267260   HD현대일렉트릭   29270273620000
22  051910       LG화학   27389829084000

In [4]:
weights_df

,종목명,투자비중,투자비중(%)
3,삼성바이오로직스,1.941877e-01,1.941877e+01
11,셀트리온,1.279570e-01,1.279570e+01
15,신한지주,1.199619e-01,1.199619e+01
23,현대모비스,1.091726e-01,1.091726e+01
4,삼성전자우,1.082722e-01,1.082722e+01
13,NAVER,7.451415e-02,7.451415e+00
29,삼성화재,5.877404e-02,5.877404e+00
17,한국전력,5.493937e-02,5.493937e+00
6,HD현대중공업,4.913567e-02,4.913567e+00
10,기아,4.231050e-02,4.231050e+00
